In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print("All libraries imported successfully.")

All libraries imported successfully.


In [11]:
# Cell 2: Load the Three Core Datasets
cities_df = pd.read_csv('../../data/broken/worldcities.csv')
cost_df = pd.read_csv('../../data/broken/costofliving.csv') 
weather_df = pd.read_csv('../../data/broken/average_weather.csv') 

print("--- DATASET SHAPES ---")
print(f"1. World Cities: {cities_df.shape}")
print(f"2. Cost of Living: {cost_df.shape}")
print(f"3. Weather Data: {weather_df.shape}")

--- DATASET SHAPES ---
1. World Cities: (48059, 11)
2. Cost of Living: (578, 8)
3. Weather Data: (465, 16)


In [12]:
# Cell 3: Diagnostics - Nulls and Data Previews
datasets = {
    "World Cities": cities_df,
    "Cost of Living": cost_df,
    "Weather Data": weather_df
}

for name, df in datasets.items():
    print(f"========== {name.upper()} ==========")
    print(f"Total Rows: {len(df)}")
    
    print("\n--- Missing Values ---")
    # Only show columns that actually have missing data to keep the output clean
    nulls = df.isnull().sum()
    missing = nulls[nulls > 0].sort_values(ascending=False)
    if not missing.empty:
        print(missing.head())
    else:
        print("Zero missing values! Clean dataset.")
        
    print("\n--- First 2 Rows ---")
    display(df.head(2))
    print("\n" + "="*50 + "\n")

========== WORLD CITIES ==========
Total Rows: 48059

--- Missing Values ---
capital       32921
population      251
admin_name      201
iso2             33
city_ascii        2
dtype: int64

--- First 2 Rows ---


,city,city_ascii,lat,lng,country,iso2,iso3,admin_name,capital,population,id
0,Tokyo,Tokyo,35.687,139.7495,Japan,JP,JPN,Tōkyō,primary,37785000.0,1392685764
1,Jakarta,Jakarta,-6.175,106.8275,Indonesia,ID,IDN,Jakarta,primary,33756000.0,1360771077




========== COST OF LIVING ==========
Total Rows: 578

--- Missing Values ---
Rank    578
dtype: int64

--- First 2 Rows ---


,Rank,City,Cost of Living Index,Rent Index,Cost of Living Plus Rent Index,Groceries Index,Restaurant Price Index,Local Purchasing Power Index
0,NaN,"Hamilton, Bermuda",149.02,96.10,124.22,157.89,155.22,79.43
1,NaN,"Zurich, Switzerland",131.24,69.26,102.19,136.14,132.52,129.79




========== WEATHER DATA ==========
Total Rows: 465

--- Missing Values ---
Ref.    2
dtype: int64

--- First 2 Rows ---


,Country,City,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Year,Ref.
0,Algeria,Algiers,11.2\n(52.2),11.9\n(53.4),12.8\n(55.0),14.7\n(58.5),17.7\n(63.9),21.3\n(70.3),24.6\n(76.3),25.2\n(77.4),23.2\n(73.8),19.4\n(66.9),15.2\n(59.4),12.1\n(53.8),17.4\n(63.3),[1]
1,Algeria,Tamanrasset,12.8\n(55.0),15.0\n(59.0),18.1\n(64.6),22.2\n(72.0),26.1\n(79.0),28.9\n(84.0),28.7\n(83.7),28.2\n(82.8),26.5\n(79.7),22.4\n(72.3),17.3\n(63.1),13.9\n(57.0),21.7\n(71.1),[2]


In [13]:
# Cell 4: Clean the Cost of Living Dataset

# 1. Drop the completely null 'Rank' column
clean_cost_df = cost_df.drop(columns=['Rank']).copy()

# 2. Split "City, Country" into two separate columns
# We use rsplit(n=1) to split only on the last comma, protecting cities with commas in their names
clean_cost_df[['city', 'country']] = clean_cost_df['City'].str.rsplit(', ', n=1, expand=True)

# 3. Clean up the remnants
clean_cost_df.drop(columns=['City'], inplace=True)
clean_cost_df['city'] = clean_cost_df['city'].str.strip()
clean_cost_df['country'] = clean_cost_df['country'].str.strip()

print("--- CLEANED COST OF LIVING ---")
print(f"Shape: {clean_cost_df.shape}")
display(clean_cost_df[['city', 'country', 'Cost of Living Index']].head(3))

--- CLEANED COST OF LIVING ---
Shape: (578, 8)


,city,country,Cost of Living Index
0,Hamilton,Bermuda,149.02
1,Zurich,Switzerland,131.24
2,Basel,Switzerland,130.93


In [15]:
# Cell 5: Clean the Weather Data

# 1. Drop the useless Reference column
clean_weather_df = weather_df.drop(columns=['Ref.']).copy()

# 2. Define the columns containing the messy strings
temp_cols = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Year']

# 3. Split the string, fix the fake minus sign, and convert to float
for col in temp_cols:
    clean_weather_df[col] = (
        clean_weather_df[col]
        .astype(str)
        .str.split('\n').str[0]       # Keep only the Celsius part
        .str.replace('−', '-')        # Replace typographic minus with standard keyboard minus
        .str.strip()                  # Remove any lingering spaces
        .astype(float)                # Convert safely to float!
    )

print("--- CLEANED WEATHER DATA ---")
print(f"Shape: {clean_weather_df.shape}")
display(clean_weather_df[['Country', 'City', 'Jul', 'Year']].head(3))

# Checking the datatypes to prove the conversion worked
print("\n--- DATA TYPES ---")
print(clean_weather_df[['Jul', 'Year']].dtypes)

--- CLEANED WEATHER DATA ---
Shape: (465, 15)


,Country,City,Jul,Year
0,Algeria,Algiers,24.6,17.4
1,Algeria,Tamanrasset,28.7,21.7
2,Algeria,Reggane,39.8,28.3



--- DATA TYPES ---
Jul     float64
Year    float64
dtype: object


In [18]:
# Cell 6: Extract the Top 250 Target Destinations

# 1. Drop rows without population data and sort from largest to smallest
top_cities_df = cities_df.dropna(subset=['population']).sort_values(by='population', ascending=False)

# 2. Slice off exactly the top 250 cities
target_destinations = top_cities_df.head(600).copy()

# 3. Keep only the columns we actually need for the project
target_destinations = target_destinations[['city_ascii', 'country', 'lat', 'lng', 'population']]

# 4. Rename 'city_ascii' to 'city' to create a perfect match key for our other datasets
target_destinations.rename(columns={'city_ascii': 'city'}, inplace=True)

print("--- TARGET DESTINATIONS (THE ANCHOR) ---")
print(f"Shape: {target_destinations.shape}")
display(target_destinations.head(3))

--- TARGET DESTINATIONS (THE ANCHOR) ---
Shape: (600, 5)


,city,country,lat,lng,population
0,Tokyo,Japan,35.687,139.7495,37785000.0
1,Jakarta,Indonesia,-6.175,106.8275,33756000.0
2,Delhi,India,28.610,77.2300,32226000.0


In [19]:
# Cell 7: The Intersection Test (Dry Run)

# 1. Create a standardized, lowercase matching column for all three dataframes
target_destinations['match_name'] = target_destinations['city'].str.lower()
clean_cost_df['match_name'] = clean_cost_df['city'].str.lower()
clean_weather_df['match_name'] = clean_weather_df['City'].str.lower()

# 2. Extract those columns into Python Sets
cities_set = set(target_destinations['match_name'])
cost_set = set(clean_cost_df['match_name'])
weather_set = set(clean_weather_df['match_name'])

# 3. The Intersection: Find the cities that exist in ALL THREE sets
survivors = cities_set & cost_set & weather_set

# 4. The Reveal
print("========== THE DRY RUN RESULTS ==========")
print(f"Target Cities Started With: {len(cities_set)}")
print(f"Cities Available in Cost Data: {len(cost_set)}")
print(f"Cities Available in Weather Data: {len(weather_set)}")
print("-" * 41)
print(f"🏆 TOTAL SURVIVORS (Perfect Match): {len(survivors)} 🏆")
print("-" * 41)

# Print a few of the winning cities just to verify they look correct
print(f"\nSample of surviving cities: {list(survivors)[:10]}")

========== THE DRY RUN RESULTS ==========
Target Cities Started With: 592
Cities Available in Cost Data: 574
Cities Available in Weather Data: 461
-----------------------------------------
🏆 TOTAL SURVIVORS (Perfect Match): 103 🏆
-----------------------------------------

Sample of surviving cities: ['jakarta', 'madrid', 'beirut', 'muscat', 'addis ababa', 'lahore', 'cairo', 'monterrey', 'bangkok', 'vancouver']


In [ ]:
# Cell 8: The Grand Merge (Execution)

# 1. Inner join target cities with the cost data 
# We drop 'city' and 'country' from the cost data before merging to prevent duplicate columns
master_df = target_destinations.merge(
    clean_cost_df.drop(columns=['city', 'country']), 
    on='match_name', 
    how='inner'
)

# 2. Inner join the result with the weather data
# We only need the July and Year averages for our feature engineering
master_df = master_df.merge(
    clean_weather_df[['match_name', 'Jul', 'Year']].rename(columns={'Jul': 'Temp_July', 'Year': 'Temp_YearAvg'}),
    on='match_name',
    how='inner'
)

# 3. Final Cleanup
# Drop the temporary lowercase matching column and reset the index so it counts cleanly from 0 to 102
master_df.drop(columns=['match_name'], inplace=True)
master_df.reset_index(drop=True, inplace=True)

print("========== FINAL DATASET COMPLETED ==========")
print(f"Final Shape: {master_df.shape}")
display(master_df.head())

========== FINAL DATASET COMPLETED ==========
Final Shape: (108, 13)


,city,country,lat,lng,population,Cost of Living Index,Rent Index,Cost of Living Plus Rent Index,Groceries Index,Restaurant Price Index,Local Purchasing Power Index,Temp_July,Temp_YearAvg
0,Tokyo,Japan,35.6870,139.7495,37785000.0,85.61,42.71,65.50,94.94,52.26,88.58,25.0,15.4
1,Jakarta,Indonesia,-6.1750,106.8275,33756000.0,40.86,16.46,29.42,41.78,23.44,25.32,26.4,26.7
2,Guangzhou,China,23.1300,113.2600,26940000.0,41.11,21.43,31.88,43.52,27.08,70.13,28.9,22.4
3,Mumbai,India,19.0761,72.8775,24973000.0,29.33,19.72,24.82,29.73,25.17,48.03,27.6,27.1
4,Manila,Philippines,14.5958,120.9772,24922000.0,40.77,25.18,33.46,39.85,25.18,22.24,28.5,28.4
